# SolarVoix — Simulation of a Critical Call

**Notebook Fabric :** `Test_Simulation_Appel_Critique`

**Lakehouse requis :** attachez `LH_SolarVoix`as the default Lakehouse.

Run initialization first, then create`CALL_0121.txt`to test the OneLake trigger.


## Cell 1 — Initialization of the alert table


In [ ]:
# === INITIALISATION — Création préventive de la table alertes_critiques ===
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime

schema_alertes = StructType([
    StructField("call_id",             StringType(),    True),
    StructField("score_risque_churn",  IntegerType(),   True),
    StructField("sentiment_ia",        StringType(),    True),
    StructField("intent_detecte",      StringType(),    True),
    StructField("statut",              StringType(),    True),
    StructField("date_alerte",         TimestampType(), True),
])

# Crée la table Delta vide seulement si elle n’existe pas déjà
if not spark.catalog.tableExists("alertes_critiques"):
    df_vide = spark.createDataFrame([], schema_alertes)
    df_vide.write.format("delta").saveAsTable("alertes_critiques")
    print("✅ Table alertes_critiques créée (vide) — prête à recevoir des alertes.")
else:
    print("ℹ️ Table alertes_critiques déjà existante.")


## Cell 2 — Creation of CALL 0121.txt


In [ ]:
# === TEST DE SIMULATION — Création d’un nouvel appel critique ===

texte_critique = """[Appel entrant - 08/04/2026] Client : Marie Dubois.
Produit concerné : Pompe à chaleur PAC.
Transcription : Je suis extrêmement en colère. Votre pompe à chaleur est tombée en panne
pour la troisième fois ce mois-ci. J’ai des enfants en bas âge et il fait froid.
J’ai contacté votre SAV cinq fois sans résultat. Je vais résilier mon contrat dès demain
et contacter mon avocat. C’est inadmissible et je veux un remboursement complet."""

chemin_test = "Files/audio_calls/CALL_0121.txt"

if mssparkutils.fs.exists(chemin_test):
    raise RuntimeError(
        f"{chemin_test} existe déjà. Supprimez-le ou choisissez un nouvel identifiant "
        "avant de tester un événement FileCreated."
    )

mssparkutils.fs.put(chemin_test, texte_critique, overwrite=False)

print("✅ Fichier CALL_0121.txt créé dans audio_calls/")


## Cell 3 — Event processing verification


In [ ]:
# === VÉRIFICATION DU TRAITEMENT ÉVÉNEMENTIEL ===

call_id_test = "CALL_0121"

df_silver_test = (
    spark.table("silver_appels_enrichis")
    .filter(f"call_id = '{call_id_test}'")
)

df_alerte_test = (
    spark.table("alertes_critiques")
    .filter(f"call_id = '{call_id_test}'")
)

nb_silver = df_silver_test.count()
nb_alertes = df_alerte_test.count()

print(f"Lignes Silver pour {call_id_test} : {nb_silver}")
print(f"Alertes pour {call_id_test}       : {nb_alertes}")

print("\n=== ENREGISTREMENT SILVER ===")
display(df_silver_test)

print("\n=== ALERTE CRITIQUE ===")
display(df_alerte_test)

if nb_silver != 1:
    raise AssertionError(
        f"Une seule ligne Silver était attendue, résultat : {nb_silver}"
    )

if nb_alertes != 1:
    raise AssertionError(
        f"Une seule alerte était attendue, résultat : {nb_alertes}"
    )

alerte = df_alerte_test.first()

if alerte.score_risque_churn < 50:
    raise AssertionError(
        f"Le score d'alerte est insuffisant : {alerte.score_risque_churn}"
    )

print(
    f"\n✅ Validation réussie : {call_id_test} a été traité "
    f"une seule fois dans chaque table avec un score de "
    f"{alerte.score_risque_churn}/100."
)
